# Phi-3 Mini QLoRA: clean held-out evaluation

This notebook reruns the Yoda-style QLoRA experiment with a **fixed 648/72 train/test split** and compares the untouched Phi-3 Mini base model with the adapted model on the same 72 unseen examples.

**Why rerun?** The earlier notebook trained on all 720 examples, so those examples cannot be described as a held-out test set. This notebook fixes that.

Evaluation has two parts:
1. **Automatic:** BERTScore F1 (gold-reference similarity and source-semantic preservation).
2. **Blinded human rubric:** 0-2 each for semantic preservation, target-style fidelity, fluency, and instruction adherence (total 0-8).

Do not use the test set to tune epochs, rank, learning rate, prompts, or decoding after seeing results.

In [ ]:
# Kaggle/Colab setup. Restart the kernel before Run All if CUDA was already initialized.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

%pip install -q transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 bitsandbytes==0.47.0 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 numpy==2.0.2 bert-score==0.3.13

In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

SEED = 42
TEST_SIZE = 72
REPO_ID = "microsoft/Phi-3-mini-4k-instruct"

np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this 4-bit QLoRA notebook.")
print(torch.cuda.get_device_name(0))

## 1. Split the raw dataset before formatting/training

The test set remains untouched until generation/evaluation.

In [ ]:
raw = load_dataset("dvgodoy/yoda_sentences", split="train")
assert len(raw) == 720

split = raw.train_test_split(test_size=TEST_SIZE, seed=SEED, shuffle=True)
train_raw = split["train"]
test_raw = split["test"]

print("train:", len(train_raw), "test:", len(test_raw))
print(test_raw[0])

In [ ]:
def format_dataset(examples):
    # Train on translation_extra, matching the original experiment.
    output_texts = []
    for src, tgt in zip(examples["sentence"], examples["translation_extra"]):
        output_texts.append([
            {"role": "user", "content": src},
            {"role": "assistant", "content": tgt},
        ])
    return {"messages": output_texts}

train_dataset = train_raw.map(format_dataset, batched=True)
train_dataset = train_dataset.remove_columns(train_dataset.column_names[:-1])
print(train_dataset[0])

## 2. Load the 4-bit base model and capture base-model predictions

Generation is deterministic (`do_sample=False`) for an apples-to-apples comparison.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,
)

model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    device_map="cuda:0",
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Model footprint: {model.get_memory_footprint()/1e6:.1f} MB")

In [ ]:
def prompts_from_sentences(sentences):
    return [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": s}],
            tokenize=False,
            add_generation_prompt=True,
        )
        for s in sentences
    ]

@torch.inference_mode()
def generate_predictions(model, sentences, batch_size=8, max_new_tokens=64):
    prompts = prompts_from_sentences(sentences)
    outputs = []
    model.eval()
    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start+batch_size]
        enc = tokenizer(
            batch_prompts,
            add_special_tokens=False,
            return_tensors="pt",
            padding=True,
        ).to(model.device)
        input_len = enc["input_ids"].shape[1]
        gen = model.generate(
            **enc,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
        new_tokens = gen[:, input_len:]
        outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
    return [x.strip() for x in outputs]

sources = list(test_raw["sentence"])
ref_plain = list(test_raw["translation"])
ref_extra = list(test_raw["translation_extra"])

base_predictions = generate_predictions(model, sources)
for i in range(3):
    print("SOURCE:", sources[i])
    print("BASE:  ", base_predictions[i])
    print("REF:   ", ref_extra[i], "\n")

## 3. QLoRA fine-tuning on the 648-example training split

These hyperparameters are kept from the prior experiment rather than selected using the test set.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
)

# Restore right padding for training.
tokenizer.padding_side = "right"

sft_config = SFTConfig(
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=1,
    per_device_train_batch_size=16,
    auto_find_batch_size=False,
    max_length=64,
    packing=False,
    num_train_epochs=10,
    learning_rate=3e-4,
    optim="paged_adamw_8bit",
    logging_steps=10,
    logging_dir="./logs-heldout",
    output_dir="./phi3-mini-yoda-adapter-heldout",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    bf16=torch.cuda.is_bf16_supported(including_emulation=False),
)

trainer = SFTTrainer(
    model=model,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_dataset,
)

trainer.model.print_trainable_parameters()
train_result = trainer.train()
print(train_result)

In [ ]:
trainer.model.save_pretrained("./phi3-mini-yoda-adapter-heldout")

# Deterministic adapted-model predictions on the exact same test inputs.
tokenizer.padding_side = "left"
adapted_predictions = generate_predictions(trainer.model, sources)

for i in range(3):
    print("SOURCE:  ", sources[i])
    print("BASE:    ", base_predictions[i])
    print("ADAPTED: ", adapted_predictions[i])
    print("REF:     ", ref_extra[i], "\n")

## 4. Automatic metrics: BERTScore F1

We report two distinct quantities:
- **Gold-reference BERTScore:** compare each output with both provided Yoda references and keep the better F1. This is the closest automatic proxy for matching the target rewrite.
- **Source-semantic BERTScore:** compare output with the original source sentence. This checks whether style transfer preserved meaning.

Do **not** treat BERTScore as a style metric by itself; that is why the human rubric follows.

In [ ]:
from bert_score import score as bertscore

def bs_f1(cands, refs):
    # CPU avoids competing with the 4-bit Phi-3 model for T4 memory.
    _, _, f1 = bertscore(cands, refs, lang="en", device="cpu", verbose=False)
    return f1.numpy()

# Score against both gold references and keep the better match per example.
base_ref_a = bs_f1(base_predictions, ref_plain)
base_ref_b = bs_f1(base_predictions, ref_extra)
adapt_ref_a = bs_f1(adapted_predictions, ref_plain)
adapt_ref_b = bs_f1(adapted_predictions, ref_extra)

base_ref_f1 = np.maximum(base_ref_a, base_ref_b)
adapt_ref_f1 = np.maximum(adapt_ref_a, adapt_ref_b)

base_sem_f1 = bs_f1(base_predictions, sources)
adapt_sem_f1 = bs_f1(adapted_predictions, sources)

metrics = pd.DataFrame({
    "source": sources,
    "reference_plain": ref_plain,
    "reference_extra": ref_extra,
    "base_output": base_predictions,
    "adapted_output": adapted_predictions,
    "base_reference_bertscore_f1": base_ref_f1,
    "adapted_reference_bertscore_f1": adapt_ref_f1,
    "base_source_semantic_bertscore_f1": base_sem_f1,
    "adapted_source_semantic_bertscore_f1": adapt_sem_f1,
})
metrics.to_csv("phi3_heldout_metrics.csv", index=False)

summary = pd.DataFrame({
    "metric": ["Gold-reference BERTScore F1", "Source-semantic BERTScore F1"],
    "base_mean": [base_ref_f1.mean(), base_sem_f1.mean()],
    "adapted_mean": [adapt_ref_f1.mean(), adapt_sem_f1.mean()],
})
summary["delta_adapted_minus_base"] = summary["adapted_mean"] - summary["base_mean"]
summary

## 5. Blinded 0-8 human rubric

Score **A and B without looking at the model key**.

Each dimension is 0, 1, or 2:

**Semantic preservation**
- 2 = same meaning; no important additions/omissions
- 1 = core meaning preserved with a minor distortion/omission
- 0 = substantial meaning changed, lost, or hallucinated

**Target-style fidelity**
- 2 = clear, consistent Yoda-like transformation
- 1 = partial/inconsistent style transformation
- 0 = little/no target style, or the wrong style

**Fluency**
- 2 = coherent and readable within the stylized syntax
- 1 = understandable but noticeably awkward/repetitive
- 0 = broken or hard to understand

**Instruction adherence**
- 2 = only a complete rewritten sentence; no meta-commentary
- 1 = minor extra text or small completion issue
- 0 = fails the rewrite task / refuses / answers instead of rewriting

Total = 0-8 per output.

For the strongest claim, score all 72 examples. If you are the only rater, state that internally as a limitation; don't call it inter-rater evaluation.

In [ ]:
# Create a blinded A/B sheet and a separate key.
rng = np.random.default_rng(SEED)
adapted_is_a = rng.random(len(sources)) < 0.5

rubric = pd.DataFrame({
    "id": np.arange(len(sources)),
    "source": sources,
    "output_A": np.where(adapted_is_a, adapted_predictions, base_predictions),
    "output_B": np.where(adapted_is_a, base_predictions, adapted_predictions),
    "A_semantic_0_2": "",
    "A_style_0_2": "",
    "A_fluency_0_2": "",
    "A_adherence_0_2": "",
    "B_semantic_0_2": "",
    "B_style_0_2": "",
    "B_fluency_0_2": "",
    "B_adherence_0_2": "",
    "notes": "",
})

key = pd.DataFrame({
    "id": np.arange(len(sources)),
    "model_A": np.where(adapted_is_a, "adapted", "base"),
    "model_B": np.where(adapted_is_a, "base", "adapted"),
})

rubric.to_csv("phi3_blind_rubric_TO_SCORE.csv", index=False)
key.to_csv("phi3_blind_rubric_key_DO_NOT_OPEN_UNTIL_DONE.csv", index=False)

print("Saved phi3_blind_rubric_TO_SCORE.csv")
print("Saved key separately. Do not open it until all scores are filled.")
rubric.head(3)

### After manually scoring the CSV

Upload the completed file back to the notebook as `phi3_blind_rubric_SCORED.csv`, then run the next cell.

In [ ]:
scored = pd.read_csv("phi3_blind_rubric_SCORED.csv")
key = pd.read_csv("phi3_blind_rubric_key_DO_NOT_OPEN_UNTIL_DONE.csv")

score_cols = [
    "A_semantic_0_2", "A_style_0_2", "A_fluency_0_2", "A_adherence_0_2",
    "B_semantic_0_2", "B_style_0_2", "B_fluency_0_2", "B_adherence_0_2",
]
for c in score_cols:
    scored[c] = pd.to_numeric(scored[c], errors="raise")
    if not scored[c].between(0, 2).all():
        raise ValueError(f"{c} contains a value outside 0..2")

scored["A_total_0_8"] = scored[[c for c in score_cols if c.startswith("A_")]].sum(axis=1)
scored["B_total_0_8"] = scored[[c for c in score_cols if c.startswith("B_")]].sum(axis=1)
joined = scored.merge(key, on="id", validate="one_to_one")

joined["base_total_0_8"] = np.where(joined["model_A"].eq("base"), joined["A_total_0_8"], joined["B_total_0_8"])
joined["adapted_total_0_8"] = np.where(joined["model_A"].eq("adapted"), joined["A_total_0_8"], joined["B_total_0_8"])

for dim in ["semantic", "style", "fluency", "adherence"]:
    joined[f"base_{dim}"] = np.where(
        joined["model_A"].eq("base"), joined[f"A_{dim}_0_2"], joined[f"B_{dim}_0_2"]
    )
    joined[f"adapted_{dim}"] = np.where(
        joined["model_A"].eq("adapted"), joined[f"A_{dim}_0_2"], joined[f"B_{dim}_0_2"]
    )

base_mean = joined["base_total_0_8"].mean()
adapt_mean = joined["adapted_total_0_8"].mean()
delta = adapt_mean - base_mean
win_rate = (joined["adapted_total_0_8"] > joined["base_total_0_8"]).mean()
tie_rate = (joined["adapted_total_0_8"] == joined["base_total_0_8"]).mean()

# Paired bootstrap CI for the mean total-score improvement.
diffs = (joined["adapted_total_0_8"] - joined["base_total_0_8"]).to_numpy()
rng = np.random.default_rng(SEED)
boot = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(10000)])
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])

print(f"Base rubric mean:    {base_mean:.2f}/8")
print(f"Adapted rubric mean: {adapt_mean:.2f}/8")
print(f"Mean delta:           {delta:+.2f} points (95% bootstrap CI {ci_lo:+.2f} to {ci_hi:+.2f})")
print(f"Adapted wins:         {100*win_rate:.1f}%")
print(f"Ties:                 {100*tie_rate:.1f}%")

per_dim = pd.DataFrame({
    "dimension": ["semantic", "style", "fluency", "adherence"],
    "base_mean": [joined[f"base_{d}"].mean() for d in ["semantic", "style", "fluency", "adherence"]],
    "adapted_mean": [joined[f"adapted_{d}"].mean() for d in ["semantic", "style", "fluency", "adherence"]],
})
per_dim["delta"] = per_dim["adapted_mean"] - per_dim["base_mean"]
per_dim

## 6. What to put on the resume after the results exist

The current resume version uses the conservative, result-neutral wording:

> Fine-tuned Phi-3 Mini (3.8B) with 4-bit NF4 QLoRA using TRL/PEFT; trained rank-8 adapters (12.6M parameters, 0.33% trainable) and compared base vs. adapted models on an unseen split using BERTScore F1 plus a rubric for meaning, style, fluency, and instruction adherence.

That is defensible once you have completed this notebook and the rubric.

If the adapted model shows a clear positive result, replace it with a quantified version. Example structure (do not fill numbers until measured):

> ... evaluated on 72 held-out examples, improving the blinded rubric from **X/8 to Y/8** while maintaining source-semantic BERTScore F1 of **Z**.

Prefer the rubric improvement as the headline because BERTScore alone does not measure Yoda-style fidelity.